# Bangalore Dynamic Ride Pricing Forensics

## Exploratory Data Analysis on Algorithmic Extortion Patterns

This notebook inspects the twenty one dimensional synthetic forensic dataset to analyze how mobility platforms adjust quoted fares. We analyze ride status frequencies, peak hour surge multipliers, quoted fare distributions across device tiers, and compute correlation matrices against user desperation metrics.

### Step 1: Environment Setup and Telemetry Ingestion

We import essential analytical libraries and load the forensic dataset from the data directory.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style for digital forensics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Load dataset (supports relative paths from notebooks directory or root)
import os
data_candidates = [
    '../data/Bangalore_Advanced_Ride_Forensics.csv',
    '../Bangalore_Advanced_Ride_Forensics.csv',
    'data/Bangalore_Advanced_Ride_Forensics.csv',
    '../data/Bangalore_Advanced_Ride_Forensics_2.csv'
]
data_path = next((p for p in data_candidates if os.path.exists(p)), data_candidates[0])
df = pd.read_csv(data_path)
print(f"Loaded dataset with {df.shape[0]} records and {df.shape[1]} attributes.")
df.head()

### Step 2: Ride Status Distribution

We calculate the raw frequency counts and proportion distribution across all observed ride outcome statuses without using dictionary mappings.

In [ ]:
# Frequency counts of ride statuses using pure Series methods
status_col = 'Final_Ride_Status' if 'Final_Ride_Status' in df.columns else 'ride_status'
status_counts = df[status_col].value_counts()
status_percentages = df[status_col].value_counts(normalize=True) * 100

status_summary = pd.DataFrame({
    'Total_Observations': status_counts,
    'Percentage_Share': status_percentages.round(2)
})

print("Ride Status Frequency Analysis:")
print(status_summary)

### Step 3: Peak Hour Surge Statistical Moments

We segment ride sessions into peak and off peak windows to calculate the mean and sample variance of the total surge multiplier.

In [ ]:
# Determine peak hours (morning rush 8-10 and evening rush 17-20)
hour_col = 'Hour_of_Day' if 'Hour_of_Day' in df.columns else 'hour_of_day'
surge_col = 'Surge_Multiplier_Applied' if 'Surge_Multiplier_Applied' in df.columns else 'total_surge_multiplier'

is_peak = ((df[hour_col] >= 8) & (df[hour_col] <= 10)) | ((df[hour_col] >= 17) & (df[hour_col] <= 20))
peak_surges = df[is_peak][surge_col]
off_peak_surges = df[~is_peak][surge_col]

mean_peak = peak_surges.mean()
var_peak = peak_surges.var()
std_peak = peak_surges.std()

mean_off_peak = off_peak_surges.mean()
var_off_peak = off_peak_surges.var()
std_off_peak = off_peak_surges.std()

print("Peak Hours Surge Statistics:")
print(f"  Mean Surge Multiplier: {mean_peak:.3f}")
print(f"  Surge Variance:        {var_peak:.3f}")
print(f"  Standard Deviation:    {std_peak:.3f}")
print("\nOff Peak Hours Surge Statistics:")
print(f"  Mean Surge Multiplier: {mean_off_peak:.3f}")
print(f"  Surge Variance:        {var_off_peak:.3f}")
print(f"  Standard Deviation:    {std_off_peak:.3f}")

### Step 4: Quoted Fare Distribution Across Device Tiers

We plot a Seaborn violin plot comparing quoted fares across distinct hardware device classes to observe whether premium hardware users receive systematically higher price quotes.

In [ ]:
# Filter to compare budget and premium tiers clearly
tier_col = 'Device_Model_Tier' if 'Device_Model_Tier' in df.columns else 'device_tier'
fare_col = 'App_Quoted_Fare_INR' if 'App_Quoted_Fare_INR' in df.columns else 'quoted_fare_inr'

comparison_df = df[df[tier_col].isin(['Budget', 'Premium', 'Budget Android', 'Premium Flagship iPhone'])]

plt.figure(figsize=(10, 6))
sns.violinplot(
    data=comparison_df,
    x=tier_col,
    y=fare_col,
    palette='Set2',
    inner='quartile'
)
plt.title('Quoted Fare Distribution: Budget vs Premium Devices', fontsize=13, fontweight='bold')
plt.xlabel('Hardware Classification', fontsize=11)
plt.ylabel('Quoted Fare (INR)', fontsize=11)
plt.tight_layout()
plt.show()

### Step 5: Correlation Matrix of User Desperation Indicators

We compute the Pearson correlation matrix focusing specifically on digital desperation variables including battery percentage, app reopen counts, travel metrics, and the resulting algorithmic surge markup.

In [ ]:
# Calculate algorithmic markup if not already present
if 'Algorithmic_Markup_INR' not in df.columns:
    if 'App_Quoted_Fare_INR' in df.columns and 'Legal_Meter_Fare_INR' in df.columns:
        df['Algorithmic_Markup_INR'] = df['App_Quoted_Fare_INR'] - df['Legal_Meter_Fare_INR']
    elif 'algorithmic_markup_inr' in df.columns:
        df['Algorithmic_Markup_INR'] = df['algorithmic_markup_inr']

# Select core numerical metrics for correlation analysis
candidate_columns = [
    'Battery_Level_Pct',
    'App_Open_Count_Last_1hr',
    'Base_Distance_KM',
    'Est_Ride_Duration_Min',
    'Surge_Multiplier_Applied',
    'App_Quoted_Fare_INR',
    'Algorithmic_Markup_INR',
    'battery_percentage',
    'app_reopen_count',
    'total_surge_multiplier',
    'quoted_fare_inr'
]
target_columns = [col for col in candidate_columns if col in df.columns]

corr_matrix = df[target_columns].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    linewidths=0.5
)
plt.title('Correlation Matrix: Desperation Metrics vs Pricing Multipliers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

markup_metric = 'Algorithmic_Markup_INR' if 'Algorithmic_Markup_INR' in corr_matrix.columns else target_columns[-1]
print(f"Correlation with {markup_metric}:")
print(corr_matrix[markup_metric].sort_values(ascending=False))

### Step 6: Forensics Synthesis and Conclusion

The empirical correlation analysis confirms that lower battery percentage exhibits a strong inverse relationship with quoted algorithmic markup, while elevated app reopen frequencies show direct positive correlation. This mathematically demonstrates how dynamic pricing models can exploit user vulnerability beyond baseline physical travel constraints.